In [12]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [13]:
from woc.remote import WocMapsRemote 
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['mmorise/World']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00144f85329256bf24124729a0554f76fda95877,mmorise_world


In [14]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

100%|██████████| 35/35 [00:35<00:00,  1.02s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00144f85329256bf24124729a0554f76fda95877,116d1fe9034b233f623863ccaac7f9b597013d68,[f254d1c0a5e9e80b9f7e8d3b4709b9e93d52f618],mmorise <mmorise@yamanashi.ac.jp>,1492872352,+0900,mmorise <mmorise@yamanashi.ac.jp>,1492872352,+0900,DC component is removed\n,00144f85329256bf24124729a0554f76fda95877,mmorise_world
1,001cffd6e178dce0cb9dc56a1c38e0a4aa6edb8a,cd51ae757e60e0e077f9b7ee28c219dcfb31cbb1,"[d2ff2b9e5210f2c898f470749fc34d96e071372b, d8a...",mmorise <masanori.morise@gmail.com>,1457054604,+0900,mmorise <masanori.morise@gmail.com>,1457054604,+0900,Merge pull request #10 from giuliopaci/master\...,001cffd6e178dce0cb9dc56a1c38e0a4aa6edb8a,mmorise_world


In [15]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00144f85329256bf24124729a0554f76fda95877', 'tree': '116d1fe9034b233f623863ccaac7f9b597013d68', 'parent': ['f254d1c0a5e9e80b9f7e8d3b4709b9e93d52f618'], 'author': 'mmorise <mmorise@yamanashi.ac.jp>', 'author_time': 1492872352, 'author_tz': '+0900', 'committer': 'mmorise <mmorise@yamanashi.ac.jp>', 'committer_time': 1492872352, 'committer_tz': '+0900', 'message': 'DC component is removed\n'}


In [16]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [17]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,mmorise_world,00144f85329256bf24124729a0554f76fda95877,mmorise <mmorise@yamanashi.ac.jp>,1492872352,DC component is removed\n


In [18]:
#mode a means append, so you have all your projects in the same file
yournetid='econstan'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [19]:
# 1. loop through all project github repos

project_names = [
    "aidenlab/straw",
    "corybrunson/ggalluvial",
    "snowleopard/alga",
    "maximecb/gym-minigrid",
    "seung-lab/cloud-volume",
    "STVIR/pysot",
    "google-research/torchsde",
    "tidymodels/rsample",
    "coin-or/Ipopt",
    "OHDSI/PatientLevelPrediction",
]

github_links = [
    "https://github.com/aidenlab/straw",
    "https://github.com/corybrunson/ggalluvial",
    "https://github.com/snowleopard/alga",
    "https://github.com/maximecb/gym-minigrid",
    "https://github.com/seung-lab/cloud-volume",
    "https://github.com/STVIR/pysot",
    "https://github.com/google-research/torchsde",
    "https://github.com/tidymodels/rsample",
    "https://github.com/coin-or/Ipopt",
    "https://github.com/OHDSI/PatientLevelPrediction",
]

# 2. report # of stars, # forks, last commit date

# 3. report # of commits, # authors, max time, min time

# 4. make CSV of projects information